# 🛒 E-Commerce Orders EDA + Machine Learning (2023–2026)

**Dataset:** 30,000 orders across 10 countries & 8 product categories  
**Goal:** End-to-end analysis — EDA, customer insights, revenue analysis, and ML prediction models

| Feature | Detail |
|---------|--------|
| Rows | 30,000 orders |
| Columns | 41 features |
| Countries | 10 (India, USA, UK, UAE, Germany, etc.) |
| Product Categories | 8 (Fashion, Electronics, Groceries, etc.) |
| Period | 2023–2026 |
| ML Tasks | Classification (High-Value Order) + Regression (Order Amount) |

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             mean_absolute_error, r2_score, confusion_matrix)

import os
os.makedirs('outputs', exist_ok=True)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')


## 1. Load & Preview Data

In [ ]:
df = pd.read_csv(r"D:\ecommerce_orders_eda\ecommerce_orders_dataset.csv")
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


In [ ]:
print("Missing Values:")
print(df.isnull().sum().sum(), "total nulls")
print()
print("Data Types:")
print(df.dtypes.value_counts())


## 2. Statistical Summary

In [ ]:
df[['Customer_Age','Unit_Price','Quantity','Discount_Percent',
     'Order_Amount','Delivery_Days','Review_Rating',
     'Customer_Lifetime_Value','Profit_Margin_Percent','Profit_Amount']].describe().T  .style.background_gradient(cmap='Blues').format(precision=2)


## 3. Revenue & Order Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Order Amount distribution
axes[0].hist(df['Order_Amount'], bins=50, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Order_Amount'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean: ${df['Order_Amount'].mean():.0f}")
axes[0].axvline(df['Order_Amount'].median(), color='yellow', linestyle='--', lw=2,
                label=f"Median: ${df['Order_Amount'].median():.0f}")
axes[0].set_title('Order Amount Distribution', fontweight='bold')
axes[0].set_xlabel('Order Amount ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Revenue by Product Category
rev_cat = df.groupby('Product_Category')['Order_Amount'].sum().sort_values(ascending=False)
colors_c = sns.color_palette('viridis', len(rev_cat))
bars = axes[1].bar(rev_cat.index, rev_cat.values/1000, color=colors_c, edgecolor='white')
axes[1].set_title('Total Revenue by Product Category ($K)', fontweight='bold')
axes[1].set_ylabel('Revenue ($K)')
axes[1].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, rev_cat.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()+5,
                 f'${val/1000:.0f}K', ha='center', va='bottom', fontsize=9)

# Avg order by Country
avg_country = df.groupby('Country')['Order_Amount'].mean().sort_values(ascending=False)
colors_co = sns.color_palette('plasma', len(avg_country))
axes[2].barh(avg_country.index[::-1], avg_country.values[::-1], color=colors_co[::-1])
axes[2].set_title('Avg Order Amount by Country ($)', fontweight='bold')
axes[2].set_xlabel('Avg Order Amount ($)')
for i, val in enumerate(avg_country.values[::-1]):
    axes[2].text(val+1, i, f'${val:.0f}', va='center', fontsize=9)

plt.suptitle('Revenue & Order Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/revenue_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# Revenue trend over time
df['YearMonth'] = df['Order_Date'].dt.to_period('M')
monthly_rev = df.groupby('YearMonth')['Order_Amount'].sum().reset_index()
monthly_rev['YearMonth_str'] = monthly_rev['YearMonth'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(22, 7))

axes[0].plot(range(len(monthly_rev)), monthly_rev['Order_Amount']/1000,
             color='#3498db', linewidth=2.5)
axes[0].fill_between(range(len(monthly_rev)), monthly_rev['Order_Amount']/1000,
                     alpha=0.2, color='#3498db')
tick_positions = list(range(0, len(monthly_rev), 6))
axes[0].set_xticks(tick_positions)
axes[0].set_xticklabels(monthly_rev['YearMonth_str'].iloc[tick_positions],
                         rotation=45, ha='right')
axes[0].set_title('Monthly Revenue Trend ($K)', fontweight='bold')
axes[0].set_ylabel('Revenue ($K)')

# Revenue by Season
season_rev = df.groupby('Season')['Order_Amount'].agg(['sum','mean']).reset_index()
x = range(len(season_rev))
w = 0.35
axes[1].bar([i-w/2 for i in x], season_rev['sum']/1000, w,
            label='Total Revenue ($K)', color='#3498db', alpha=0.85)
ax2 = axes[1].twinx()
ax2.bar([i+w/2 for i in x], season_rev['mean'], w,
         label='Avg Order ($)', color='#e74c3c', alpha=0.85)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(season_rev['Season'])
axes[1].set_title('Revenue by Season', fontweight='bold')
axes[1].set_ylabel('Total Revenue ($K)', color='#3498db')
ax2.set_ylabel('Avg Order Amount ($)', color='#e74c3c')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.suptitle('Revenue Trends', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/revenue_trends.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Customer Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

# Age distribution
axes[0].hist(df['Customer_Age'], bins=40, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Customer_Age'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean: {df['Customer_Age'].mean():.0f} yrs")
axes[0].set_title('Customer Age Distribution', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# Gender split
gen = df['Customer_Gender'].value_counts()
axes[1].pie(gen.values, labels=gen.index, autopct='%1.1f%%',
            colors=['#3498db','#e74c3c'], startangle=90, textprops={'fontsize':13})
axes[1].set_title('Gender Distribution', fontweight='bold')

# Membership status
mem = df['Membership_Status'].value_counts()
colors_m = ['#95a5a6','#f39c12','#f1c40f','#2ecc71']
bars = axes[2].bar(mem.index, mem.values, color=colors_m, edgecolor='white')
axes[2].set_title('Membership Status Distribution', fontweight='bold')
axes[2].set_ylabel('Count')
for bar, val in zip(bars, mem.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 f'{val:,}', ha='center', va='bottom', fontsize=11)

# CLV by Membership
mem_order = ['Standard','Silver','Gold','Platinum']
clv_mem = df.groupby('Membership_Status')['Customer_Lifetime_Value'].mean().reindex(mem_order)
bars2 = axes[3].bar(mem_order, clv_mem.values, color=colors_m, edgecolor='white')
axes[3].set_title('Avg CLV by Membership Status ($)', fontweight='bold')
axes[3].set_ylabel('Avg Customer Lifetime Value ($)')
for bar, val in zip(bars2, clv_mem.values):
    axes[3].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 f'${val:.0f}', ha='center', va='bottom', fontsize=11)

# Customer segment
seg = df.groupby('Customer_Segment')['Order_Amount'].agg(['mean','count']).reset_index()
axes[4].bar(seg['Customer_Segment'], seg['mean'],
            color=sns.color_palette('Set2', len(seg)), edgecolor='white')
axes[4].set_title('Avg Order Amount by Customer Segment ($)', fontweight='bold')
axes[4].set_ylabel('Avg Order Amount ($)')
for i, (_, row) in enumerate(seg.iterrows()):
    axes[4].text(i, row['mean']+1, f'${row["mean"]:.0f}', ha='center', va='bottom', fontsize=10)

# Avg order by age group
df['Age_Group'] = pd.cut(df['Customer_Age'], bins=[17,30,40,50,60,70],
                          labels=['18-30','31-40','41-50','51-60','61-70'])
age_rev = df.groupby('Age_Group', observed=True)['Order_Amount'].mean()
axes[5].bar(age_rev.index, age_rev.values,
            color=sns.color_palette('coolwarm', len(age_rev)), edgecolor='white')
axes[5].set_title('Avg Order Amount by Age Group ($)', fontweight='bold')
axes[5].set_ylabel('Avg Order Amount ($)')
for i, val in enumerate(age_rev.values):
    axes[5].text(i, val+1, f'${val:.0f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Customer Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/customer_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 5. Product & Category Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Avg order by product category
avg_cat = df.groupby('Product_Category')['Order_Amount'].mean().sort_values(ascending=False)
colors_cat = sns.color_palette('tab10', len(avg_cat))
bars = axes[0].bar(avg_cat.index, avg_cat.values, color=colors_cat, edgecolor='white')
axes[0].set_title('Avg Order Amount by Category ($)', fontweight='bold')
axes[0].set_ylabel('Avg Order ($)')
axes[0].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, avg_cat.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'${val:.0f}', ha='center', va='bottom', fontsize=9)

# Return rate by category
ret_cat = df.groupby('Product_Category')['Returned'].apply(
    lambda x: (x=='Yes').mean()*100).sort_values(ascending=False)
colors_ret = ['#e74c3c' if v > ret_cat.mean() else '#2ecc71' for v in ret_cat.values]
bars2 = axes[1].bar(ret_cat.index, ret_cat.values, color=colors_ret, edgecolor='white')
axes[1].axhline(ret_cat.mean(), color='gray', linestyle='--', lw=1.5,
                label=f'Avg: {ret_cat.mean():.1f}%')
axes[1].set_title('Return Rate by Category (%)', fontweight='bold')
axes[1].set_ylabel('Return Rate (%)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

# Avg review rating by category
rat_cat = df.groupby('Product_Category')['Review_Rating'].mean().sort_values(ascending=False)
colors_rat = sns.color_palette('RdYlGn', len(rat_cat))
axes[2].barh(rat_cat.index[::-1], rat_cat.values[::-1], color=colors_rat[::-1])
axes[2].set_title('Avg Review Rating by Category', fontweight='bold')
axes[2].set_xlabel('Avg Rating (1-5)')
for i, val in enumerate(rat_cat.values[::-1]):
    axes[2].text(val+0.01, i, f'{val:.2f}★', va='center', fontsize=10)

plt.suptitle('Product & Category Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/product_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 6. Payment Method & Traffic Source Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Payment method distribution
pay = df['Payment_Method'].value_counts()
colors_pay = sns.color_palette('Set2', len(pay))
axes[0].pie(pay.values, labels=pay.index, autopct='%1.1f%%',
            colors=colors_pay, startangle=90, textprops={'fontsize':10})
axes[0].set_title('Payment Method Distribution', fontweight='bold')

# Avg order by traffic source
traffic = df.groupby('Traffic_Source')['Order_Amount'].mean().sort_values(ascending=False)
colors_tr = sns.color_palette('plasma', len(traffic))
bars = axes[1].bar(traffic.index, traffic.values, color=colors_tr, edgecolor='white')
axes[1].set_title('Avg Order Amount by Traffic Source ($)', fontweight='bold')
axes[1].set_ylabel('Avg Order Amount ($)')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, traffic.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'${val:.0f}', ha='center', va='bottom', fontsize=10)

# Device type
dev = df.groupby('Device_Type')['Order_Amount'].agg(['mean','count']).reset_index()
axes[2].bar(dev['Device_Type'], dev['mean'],
            color=['#e74c3c','#3498db','#2ecc71'], edgecolor='white')
axes[2].set_title('Avg Order Amount by Device Type ($)', fontweight='bold')
axes[2].set_ylabel('Avg Order Amount ($)')
for i, row in dev.iterrows():
    label_str = f'${row["mean"]:.0f}' + chr(10) + f'({row["count"]:,})'
    axes[2].text(i, row['mean']+0.5, label_str,
                 ha='center', va='bottom', fontsize=10)

plt.suptitle('Payment Method & Traffic Source Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/payment_traffic.png', bbox_inches='tight', dpi=150)
plt.show()


## 7. Shipping & Delivery Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Order status distribution
status = df['Order_Status'].value_counts()
colors_st = ['#2ecc71','#3498db','#e74c3c','#f39c12','#9b59b6']
axes[0].bar(status.index, status.values, color=colors_st, edgecolor='white')
axes[0].set_title('Order Status Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, val in enumerate(status.values):
    axes[0].text(i, val+20, f'{val:,}', ha='center', va='bottom', fontsize=10)

# Delivery days distribution
axes[1].hist(df['Delivery_Days'], bins=20, color='#e67e22', edgecolor='white', alpha=0.85)
axes[1].axvline(df['Delivery_Days'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean: {df['Delivery_Days'].mean():.1f} days")
axes[1].set_title('Delivery Days Distribution', fontweight='bold')
axes[1].set_xlabel('Delivery Days')
axes[1].set_ylabel('Count')
axes[1].legend()

# Avg delivery days by shipping method
ship = df.groupby('Shipping_Method')['Delivery_Days'].mean().sort_values()
colors_sh = sns.color_palette('RdYlGn', len(ship))
axes[2].barh(ship.index, ship.values, color=colors_sh)
axes[2].set_title('Avg Delivery Days by Shipping Method', fontweight='bold')
axes[2].set_xlabel('Avg Delivery Days')
for i, val in enumerate(ship.values):
    axes[2].text(val+0.05, i, f'{val:.1f}d', va='center', fontsize=10)

plt.suptitle('Shipping & Delivery Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/shipping_delivery.png', bbox_inches='tight', dpi=150)
plt.show()


## 8. Profit & Discount Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Profit margin distribution
axes[0].hist(df['Profit_Margin_Percent'], bins=40, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Profit_Margin_Percent'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean: {df['Profit_Margin_Percent'].mean():.1f}%")
axes[0].set_title('Profit Margin % Distribution', fontweight='bold')
axes[0].set_xlabel('Profit Margin (%)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Profit margin by category
prof_cat = df.groupby('Product_Category')['Profit_Margin_Percent'].mean().sort_values(ascending=False)
colors_pc = sns.color_palette('RdYlGn', len(prof_cat))
axes[1].barh(prof_cat.index[::-1], prof_cat.values[::-1], color=colors_pc[::-1])
axes[1].set_title('Avg Profit Margin % by Category', fontweight='bold')
axes[1].set_xlabel('Avg Profit Margin (%)')
for i, val in enumerate(prof_cat.values[::-1]):
    axes[1].text(val+0.1, i, f'{val:.1f}%', va='center', fontsize=9)

# Discount % vs Profit Amount scatter
scatter = axes[2].scatter(df['Discount_Percent'], df['Profit_Amount'],
                          c=df['Order_Amount'], cmap='viridis', alpha=0.3, s=15)
plt.colorbar(scatter, ax=axes[2], label='Order Amount ($)')
z = np.polyfit(df['Discount_Percent'], df['Profit_Amount'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, df['Discount_Percent'].max(), 100)
axes[2].plot(x_line, p(x_line), color='red', linewidth=2, linestyle='--')
axes[2].set_title('Discount % vs Profit Amount', fontweight='bold')
axes[2].set_xlabel('Discount (%)')
axes[2].set_ylabel('Profit Amount ($)')

plt.suptitle('Profit & Discount Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/profit_discount.png', bbox_inches='tight', dpi=150)
plt.show()


## 9. Correlation Heatmap

In [ ]:
num_cols = ['Customer_Age','Unit_Price','Quantity','Discount_Percent',
            'Shipping_Cost','Tax_Amount','Order_Amount','Delivery_Days',
            'Review_Rating','Customer_Lifetime_Value',
            'Profit_Margin_Percent','Profit_Amount']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5,
            annot_kws={'size':10}, square=True,
            cbar_kws={'shrink':0.8})
ax.set_title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()


## 10. Machine Learning — Classification

**Task:** Predict whether an order is High-Value (`High_Value_Order`)  
**Model:** Random Forest Classifier


In [ ]:
X = df.drop(columns=['Order_ID','Customer_ID','Order_Date','YearMonth',
                       'Age_Group','Order_Amount','High_Value_Order'])
y = df['High_Value_Order']

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', 'passthrough', num_cols)
])

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
clf.fit(X_train, y_train)
pred_clf = clf.predict(X_test)

print('='*60)
print('RANDOM FOREST CLASSIFIER — High Value Order Prediction')
print('='*60)
print(f"Accuracy  : {accuracy_score(y_test, pred_clf):.4f}")
print()
print(classification_report(y_test, pred_clf))


In [ ]:
# Feature importance
feature_names = (list(clf['preprocessor'].transformers_[0][1]
                      .get_feature_names_out(cat_cols)) + num_cols)
importances = clf['model'].feature_importances_
feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(14, 7))
colors_fi = sns.color_palette('viridis', len(feat_df))
ax.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1], color=colors_fi[::-1])
ax.set_title('Top 15 Feature Importances — High Value Order Classifier',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('outputs/feature_importance_clf.png', bbox_inches='tight', dpi=150)
plt.show()


## 11. Machine Learning — Regression

**Task:** Predict Order Amount  
**Model:** Random Forest Regressor


In [ ]:
X_reg = df.drop(columns=['Order_ID','Customer_ID','Order_Date','YearMonth',
                           'Age_Group','High_Value_Order','Order_Amount',
                           'Discount_Amount','Tax_Amount','Profit_Amount'])
y_reg = df['Order_Amount']

cat_cols_r = X_reg.select_dtypes(include='object').columns.tolist()
num_cols_r = X_reg.select_dtypes(exclude='object').columns.tolist()

preprocessor_reg = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_r),
    ('num', 'passthrough', num_cols_r)
])

reg = Pipeline(steps=[
    ('preprocessor', preprocessor_reg),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)
reg.fit(X_train_r, y_train_r)
pred_reg = reg.predict(X_test_r)

mae  = mean_absolute_error(y_test_r, pred_reg)
r2   = r2_score(y_test_r, pred_reg)

print('='*60)
print('RANDOM FOREST REGRESSOR — Order Amount Prediction')
print('='*60)
print(f"MAE      : ${mae:.2f}")
print(f"R² Score : {r2:.4f}")
print(f"RMSE     : ${np.sqrt(((y_test_r - pred_reg)**2).mean()):.2f}")


In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sample_idx = np.random.choice(len(y_test_r), 300, replace=False)
axes[0].scatter(y_test_r.values[sample_idx], pred_reg[sample_idx],
                alpha=0.5, color='#3498db', s=25)
max_val = max(y_test_r.max(), pred_reg.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_title('Actual vs Predicted Order Amount (300 sample)', fontweight='bold')
axes[0].set_xlabel('Actual Order Amount ($)')
axes[0].set_ylabel('Predicted Order Amount ($)')
axes[0].legend()

# Residuals
residuals = y_test_r.values - pred_reg
axes[1].hist(residuals, bins=50, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', lw=2)
axes[1].set_title('Prediction Residuals Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')

plt.suptitle('Regression Model — Order Amount Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/regression_results.png', bbox_inches='tight', dpi=150)
plt.show()


## 12. Key Insights Summary

In [ ]:
total_rev      = df['Order_Amount'].sum()
total_profit   = df['Profit_Amount'].sum()
avg_order      = df['Order_Amount'].mean()
top_category   = df.groupby('Product_Category')['Order_Amount'].sum().idxmax()
top_country    = df.groupby('Country')['Order_Amount'].sum().idxmax()
return_rate    = (df['Returned']=='Yes').mean()*100
avg_rating     = df['Review_Rating'].mean()
high_val_pct   = (df['High_Value_Order']=='Yes').mean()*100
top_payment    = df['Payment_Method'].value_counts().idxmax()
avg_profit_pct = df['Profit_Margin_Percent'].mean()

lines = [
    "+---------------------------------------------------------------+",
    "|        E-COMMERCE ORDERS 2023-2026 -- KEY INSIGHTS           |",
    "+---------------------------------------------------------------+",
    f"|  Total Orders         : {len(df):,}                           |",
    f"|  Total Revenue        : ${total_rev:,.0f}                  |",
    f"|  Total Profit         : ${total_profit:,.0f}                  |",
    f"|  Avg Order Amount     : ${avg_order:.2f}                         |",
    f"|  Avg Profit Margin    : {avg_profit_pct:.1f}%                              |",
    f"|  Top Revenue Category : {top_category:<28}     |",
    f"|  Top Revenue Country  : {top_country:<28}     |",
    f"|  Return Rate          : {return_rate:.1f}%                              |",
    f"|  Avg Review Rating    : {avg_rating:.2f}/5.0                          |",
    f"|  High Value Orders    : {high_val_pct:.1f}% of all orders               |",
    f"|  Top Payment Method   : {top_payment:<28}     |",
    "+---------------------------------------------------------------+",
]
print(chr(10).join(lines))


---
## EDA + ML Complete!

**Next Steps:**
- Run the Streamlit dashboard: `streamlit run app.py`
- Star this repo on GitHub!

> Built with Python, Pandas, Matplotlib, Seaborn & Scikit-learn
